In [1]:
from surprise import accuracy, Dataset, SVD, Reader
from surprise.model_selection import train_test_split
import pandas as pd
from surprise.model_selection import cross_validate
from collections import defaultdict

df_ratings_total = pd.read_csv('ml-latest-small/ratings.csv').drop(columns='timestamp')
df_ratings_traing = pd.read_csv('data_spliting_cf/training.csv').drop(columns='timestamp')
df_ratings_testing = pd.read_csv('data_spliting_cf/testing.csv').drop(columns='timestamp')
df_movies = pd.read_csv('ml-latest-small/movies.csv')

reader = Reader(rating_scale=([0.5, 5]))

# Convertendo os dois separadamente
data_training = Dataset.load_from_df(df_ratings_traing, reader=reader)
data_testing = Dataset.load_from_df(df_ratings_testing, reader=reader)

'''
O Surprise espera dois formatos diferentes:

Trainset → um objeto especial interno do Surprise, usado pelo .fit()
Testset → uma lista de tuplas (uid, iid, nota_real), usado pelo .test()

'''

# Treino: build_full_trainset() — constrói um trainset com TODOS os dados do DataFrame
trainset = data_training.build_full_trainset()

# Teste: build_full_trainset().build_testset() — converte para lista de tuplas (uid, iid, nota_real)
testset = data_testing.build_full_trainset().build_testset()


algo = SVD(    
    n_factors=200,    # dimensões do espaço latente
    n_epochs=50,      # épocas de SGD
    lr_all=0.005,     # learning rate
    reg_all=0.02,     # regularização L2
    random_state=42,
    verbose=True
)

algo.fit(trainset)

#### Algoítimo aprendido:
print(f'vetor latente usuário: {algo.pu.shape}')
print(f'vetor latente filme: {algo.qi.shape}')
print(f'vetor de vies de cada usuário: {algo.bu.shape}')
print(f'vetor de vies de cada filme: {algo.bi.shape}')


## A ideia principal do modelo é prever nota
''' 
Aqui ele preve a nota do user 1 no filme 4
'''
algo.predict(uid=1, iid=1)

predicoes = algo.test(testset)


print(len(predicoes))

# NOTA: O SVD nunca marca was_impossible=True para filmes/usuários desconhecidos —
# ele usa a média global como fallback sem lançar PredictionImpossible.
# Por isso filtramos manualmente comparando com os IDs vistos no treino.
filmes_no_treino = set(df_ratings_traing['movieId'].unique())
usuarios_no_treino = set(df_ratings_traing['userId'].unique())

impossiveis = [pred for pred in predicoes if pred.iid not in filmes_no_treino or pred.uid not in usuarios_no_treino]
possiveis = [pred for pred in predicoes if pred.iid in filmes_no_treino and pred.uid in usuarios_no_treino]

print(f"Previsões normais: {len(possiveis)}")
print(f"Previsões impossíveis: {len(impossiveis)}")

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49
vetor late

In [2]:
def recomenda_usuario(usuario_id, top_n=10):
    filmes_assistidos = df_ratings_traing[df_ratings_traing['userId'] == usuario_id]['movieId'].tolist()

    filmes_nao_assistidos = sorted([
        movie_id for movie_id in filmes_no_treino
        if movie_id not in filmes_assistidos
    ])

    predicoes = []
    for id in filmes_nao_assistidos:
        pred = algo.predict(uid=usuario_id, iid=id)
        predicoes.append((id, pred.est))

    arr_sorted = sorted(predicoes, key=lambda x: x[1], reverse=True)

    recomendados = arr_sorted[:top_n]
    ids_recomendados = [movie_id for movie_id, nota in recomendados]

    df_resultado = df_movies.loc[df_movies['movieId'].isin(ids_recomendados), ['movieId', 'titulo_movie_lens']]

    return df_resultado


recomendados = recomenda_usuario(1)


In [3]:
def precisao_recall_usuario(usuario_id, top_n=10):
    ids_recomendados = recomenda_usuario(usuario_id, top_n)['movieId'].tolist()

    filmes_no_teste_usuario = df_ratings_testing[df_ratings_testing['userId'] == usuario_id]['movieId'].tolist()

    relevantes_recomendados = [m for m in ids_recomendados if m in filmes_no_teste_usuario]

    precisao = len(relevantes_recomendados) / len(ids_recomendados) if ids_recomendados else 0
    recall = len(relevantes_recomendados) / len(filmes_no_teste_usuario) if filmes_no_teste_usuario else 0

    print(f"Usuário: {usuario_id}")
    print(f"Filmes recomendados: {len(ids_recomendados)}")
    print(f"Filmes do usuário no teste: {len(filmes_no_teste_usuario)}")
    print(f"Relevantes recomendados: {len(relevantes_recomendados)}")
    print(f"Precisão: {precisao:.4f}")
    print(f"Recall:   {recall:.4f}")

    return precisao, recall


precisao_recall_usuario(1)


Usuário: 1
Filmes recomendados: 10
Filmes do usuário no teste: 70
Relevantes recomendados: 2
Precisão: 0.2000
Recall:   0.0286


(0.2, 0.02857142857142857)

In [4]:
def precisao_recall_model(top_n=10):
    usuarios = df_ratings_testing['userId'].unique()

    precisoes = []
    recalls = []

    for usuario_id in usuarios:
        precisao, recall = precisao_recall_usuario(usuario_id, top_n)
        precisoes.append(precisao)
        recalls.append(recall)

    precisao_media = sum(precisoes) / len(precisoes)
    recall_medio = sum(recalls) / len(recalls)

    print(f"Usuários avaliados: {len(usuarios)}")
    print(f"Precisão média: {precisao_media:.4f}")
    print(f"Recall médio:   {recall_medio:.4f}")

    return precisao_media, recall_medio


precisao_recall_model()


Usuário: 1
Filmes recomendados: 10
Filmes do usuário no teste: 70
Relevantes recomendados: 2
Precisão: 0.2000
Recall:   0.0286
Usuário: 2
Filmes recomendados: 10
Filmes do usuário no teste: 9
Relevantes recomendados: 0
Precisão: 0.0000
Recall:   0.0000
Usuário: 3
Filmes recomendados: 10
Filmes do usuário no teste: 12
Relevantes recomendados: 0
Precisão: 0.0000
Recall:   0.0000
Usuário: 4
Filmes recomendados: 10
Filmes do usuário no teste: 65
Relevantes recomendados: 1
Precisão: 0.1000
Recall:   0.0154
Usuário: 5
Filmes recomendados: 10
Filmes do usuário no teste: 14
Relevantes recomendados: 0
Precisão: 0.0000
Recall:   0.0000
Usuário: 6
Filmes recomendados: 10
Filmes do usuário no teste: 95
Relevantes recomendados: 0
Precisão: 0.0000
Recall:   0.0000
Usuário: 7
Filmes recomendados: 10
Filmes do usuário no teste: 46
Relevantes recomendados: 1
Precisão: 0.1000
Recall:   0.0217
Usuário: 8
Filmes recomendados: 10
Filmes do usuário no teste: 15
Relevantes recomendados: 0
Precisão: 0.0000
Re

(0.05311475409836065, 0.01678962118787329)

In [7]:
def precision_recall_at_k(predictions, k=10, threshold=4.0):
    """Calcula Precision@K e Recall@K para todas as predições."""
    # Agrupa predições por usuário
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))


    precisions = {}
    recalls    = {}

    for uid, user_ratings in user_est_true.items():
        # Ordena por nota estimada (decrescente) e pega top-K

        print(user_ratings)
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        # Hits: entre os top-K, quantos são realmente relevantes?
        n_rel_top_k    = sum(true_r >= threshold for _, true_r in top_k)
        # Total de relevantes no conjunto de teste deste usuário
        n_rel_total    = sum(true_r >= threshold for _, true_r in user_ratings)

        precisions[uid] = n_rel_top_k / k
        recalls[uid]    = n_rel_top_k / n_rel_total if n_rel_total > 0 else 0.0

    mean_precision = sum(precisions.values()) / len(precisions)
    mean_recall    = sum(recalls.values()) / len(recalls)
    return mean_precision, mean_recall, precisions, recalls


K = 10
mean_p, mean_r, precisions, recalls = precision_recall_at_k(predicoes, k=K, threshold=3.5)

print(f'Mean Precision@{K}: {mean_p:.4f}')
print(f'Mean Recall@{K}:    {mean_r:.4f}')
print(f'Usuários avaliados: {len(precisions)}')

[(3.8121344471576815, 3.0), (4.064137656449058, 5.0), (4.171250901961878, 4.0), (5, 5.0), (4.695684684416629, 5.0), (4.463163520260393, 5.0), (5, 5.0), (4.975962317305225, 5.0), (4.985320913118437, 3.0), (4.751000416590576, 4.0), (4.2754239472316335, 5.0), (4.564734069956046, 3.0), (4.018572745135863, 5.0), (4.3060608857747855, 4.0), (4.297814435788727, 4.0), (3.8599548140595403, 4.0), (3.790115218594801, 4.0), (3.221550245509947, 2.0), (3.553577290753639, 3.0), (4.101073489739006, 5.0), (4.848018035574808, 5.0), (4.775908402353347, 4.0), (4.532144279609835, 5.0), (5, 5.0), (4.820581285865818, 2.0), (4.3138515532791635, 4.0), (4.225308987949925, 4.0), (4.365527227352492, 5.0), (5, 3.0), (4.98820839275636, 5.0), (4.136628816671298, 5.0), (4.346928371137186, 4.0), (4.5363145465595425, 5.0), (4.161964393551565, 3.0), (3.9530362067432923, 5.0), (4.44565966411834, 5.0), (3.823351188992612, 5.0), (3.924624807298598, 1.0), (2.963317427332915, 3.0), (3.399983598091012, 5.0), (3.451379287784001

In [6]:
from collections import Counter

def filmes_mais_recomendados(top_n_usuario=10, top_n_resultado=20):
    usuarios = df_ratings_traing['userId'].unique()
    todos_recomendados = []

    for usuario_id in usuarios:
        df_rec = recomenda_usuario(usuario_id, top_n_usuario)
        todos_recomendados.extend(df_rec['movieId'].tolist())

    contagem = Counter(todos_recomendados)
    mais_comuns = contagem.most_common(top_n_resultado)

    ids = [movie_id for movie_id, _ in mais_comuns]
    qtds = [qtd for _, qtd in mais_comuns]

    df_resultado = df_movies[df_movies['movieId'].isin(ids)][['movieId', 'titulo_movie_lens']].copy()
    df_resultado['vezes_recomendado'] = df_resultado['movieId'].map(dict(mais_comuns))
    df_resultado = df_resultado.sort_values('vezes_recomendado', ascending=False).reset_index(drop=True)

    return df_resultado


df_mais_recomendados = filmes_mais_recomendados(top_n_usuario=10, top_n_resultado=20)
df_mais_recomendados


,movieId,titulo_movie_lens,vezes_recomendado
0,951,His Girl Friday (1940),134
1,1283,High Noon (1952),132
2,318,"Shawshank Redemption, The (1994)",128
3,3030,Yojimbo (1961),116
4,750,Dr. Strangelove or: How I Learned to Stop Worr...,115
5,912,Casablanca (1942),113
6,1204,Lawrence of Arabia (1962),99
7,1104,"Streetcar Named Desire, A (1951)",79
8,858,"Godfather, The (1972)",75
9,2324,Life Is Beautiful (La Vita è bella) (1997),74
